# Error Handling

How to handle errors across the Jockey API, including retry strategies, polling helpers, and batch polling for async operations.

# Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

In [ ]:
import concurrent.futures
import os
import time

from twelvelabs import TwelveLabs
from twelvelabs.core.api_error import ApiError

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
STORE_ID = os.environ.get("TWELVELABS_STORE_ID", "your_store_id")  # Replace with your knowledge store ID

client = TwelveLabs(api_key=API_KEY)

## Error Response Format

All errors follow the same shape:

```json
{
  "code": "invalid_request",
  "message": "The 'method' field is required",
  "docs_url": "https://docs.twelvelabs.io/errors/invalid_request"
}
```

## HTTP Status Codes

| Code | Meaning | Action |
|------|---------|--------|
| `200` | Success | Process response |
| `201` | Created | Resource created successfully |
| `202` | Accepted | Async operation started (knowledge store items) |
| `400` | Bad Request | Fix request parameters |
| `401` | Unauthorized | Check API key |
| `403` | Forbidden | Check permissions |
| `404` | Not Found | Check resource ID |
| `429` | Rate Limited | Back off and retry |
| `500` | Server Error | Retry with backoff |

## Retry Strategy

Use exponential backoff for rate-limited (429) and server error (5xx) responses.

In [ ]:
def with_retry(call, max_retries: int = 3, base_delay: int = 1):
    """Call an SDK method with exponential backoff on transient errors.

    Retries on 429 (rate limited) and 5xx (server error). Client errors
    (400, 401, 403, 404) are not retried - fix the request instead.

    Args:
        call: A zero-argument callable that performs the SDK request.
        max_retries: Maximum number of attempts before giving up.
        base_delay: Seconds to wait before the first retry, doubling each time.

    Returns:
        Whatever `call` returns.
    """
    for attempt in range(max_retries):
        try:
            return call()
        except ApiError as exc:
            status = exc.status_code
            if status == 429 or (status is not None and status >= 500):
                delay = base_delay * (2**attempt)
                print(f"Received {status}, retrying in {delay}s...")
                time.sleep(delay)
                continue
            raise  # Client errors are not retryable
    raise RuntimeError(f"Failed after {max_retries} attempts")

In [ ]:
# Example: using the retry wrapper
response = with_retry(
    lambda: client.responses.create(
        knowledge_store_id=STORE_ID,
        input=[
            {
                "type": "message",
                "role": "user",
                "content": "Summarize these videos and images",
            }
        ],
    )
)

for output in response.output:
    if output.type == "message":
        for content in output.content:
            print(content.text)

## Common Errors by Endpoint

### Assets

| Error | Cause | Fix |
|-------|-------|-----|
| `method` required | Missing upload method | Add `method: "direct"` or `method: "url"` |
| File too large | Direct upload > 200MB | Use URL upload method instead |
| Invalid URL | URL not accessible | Check URL is public and reachable |

### Knowledge Stores

| Error | Cause | Fix |
|-------|-------|-----|
| Invalid schema | Bad JSON Schema in ingestion config | Validate against JSON Schema draft 2020-12 |
| Name required | Missing `name` field | Add a name |

### Knowledge Store Items

| Error | Cause | Fix |
|-------|-------|-----|
| Asset not found | Invalid `asset_id` | Check asset exists and is `ready` |
| Store not found | Invalid `knowledge_store_id` | Check knowledge store ID |

### Responses

| Error | Cause | Fix |
|-------|-------|-----|
| Knowledge store required | Missing `knowledge_store_id` | Add `knowledge_store_id` to the request body |
| Invalid session | Bad `session_id` | Start a new session (omit session_id) |
| Input required | Missing `input` array | Add at least one message |

## Polling and Async Status

Jockey processes content asynchronously. Any time you create an asset or add a video to a knowledge store, processing happens in the background. You must poll for completion before proceeding.

### Polling Helper

In [ ]:
def wait_for_ready(fetch, interval: int = 5, timeout: int = 600):
    """Poll a resource until it reaches 'ready' or 'failed' status.

    Args:
        fetch: A zero-argument callable returning the current resource, e.g.
            lambda: client.assets.retrieve(asset_id=asset_id)
        interval: Seconds between poll attempts.
        timeout: Seconds to wait before giving up.

    Returns:
        The resource once it is ready.

    Raises:
        Exception: If the resource reaches the 'failed' status.
        TimeoutError: If the resource is not ready before the timeout.
    """
    elapsed = 0
    while elapsed < timeout:
        resource = fetch()
        if resource.status == "ready":
            return resource
        if resource.status == "failed":
            raise Exception(f"Resource failed: {resource.id}")
        print(f"  Status: {resource.status} ({elapsed}s elapsed)")
        time.sleep(interval)
        elapsed += interval
    raise TimeoutError(f"Not ready after {timeout}s")

### Polling Usage

Use appropriate intervals for different resource types.

In [ ]:
# Example asset and item IDs (replace with real values)
ASSET_ID = "your_asset_id"
ITEM_ID = "your_item_id"

# Wait for an asset (shorter interval -- uploads finish faster)
# asset = wait_for_ready(lambda: client.assets.retrieve(asset_id=ASSET_ID), interval=5)

# Wait for a knowledge store item (longer interval -- indexing takes more time)
# item = wait_for_ready(
#     lambda: client.knowledge_store_items.retrieve(
#         knowledge_store_id=STORE_ID, item_id=ITEM_ID
#     ),
#     interval=10,
#     timeout=600,
# )

### Recommended Intervals

| Resource | Poll Interval | Typical Wait | Timeout |
|----------|--------------|-------------|--------|
| Asset (direct) | 5s | 10-60s | 120s |
| Asset (URL) | 5s | 10-120s | 300s |
| Knowledge store item | 10s | 1-10 min | 600s |

### Batch Polling

Wait for multiple resources in parallel using `concurrent.futures`.

In [ ]:
def wait_for_items(store_id: str, item_ids: list[str], max_workers: int = 5) -> dict:
    """Wait for multiple knowledge store items in parallel.

    Args:
        store_id: The knowledge store ID.
        item_ids: List of item IDs to wait for.
        max_workers: Maximum number of concurrent polling threads.

    Returns:
        A dict mapping each item ID to its ready resource.
    """

    def wait_one(item_id: str):
        return wait_for_ready(
            lambda: client.knowledge_store_items.retrieve(
                knowledge_store_id=store_id, item_id=item_id
            ),
            interval=10,
        )

    results = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(wait_one, iid): iid for iid in item_ids}
        for future in concurrent.futures.as_completed(futures):
            item_id = futures[future]
            results[item_id] = future.result()
    return results

In [ ]:
# Example: batch polling (replace with real item IDs)
# item_ids = ["item_001", "item_002", "item_003"]
# results = wait_for_items(STORE_ID, item_ids)
# for item_id, resource in results.items():
#     print(f"{item_id}: {resource.status}")

print("Uncomment the lines above with real item IDs to run batch polling.")

### Polling Pitfalls

- **Do not poll too aggressively** -- 1-second intervals waste rate limit budget
- **Always handle `failed`** -- failed resources do not recover
- **Set a timeout** -- do not poll forever if something goes wrong
- **Webhooks not available yet** -- polling is the only option in Private Beta

## Next Steps

- [Streaming](./streaming.ipynb) -- Receive responses in real-time via SSE
- [Structured Output](./structured_output.ipynb) -- Force Jockey to return typed JSON
- [Multi-Turn Sessions](./multi_turn_sessions.ipynb) -- Continue conversations across multiple requests
- [API Reference: POST /responses](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/responses/create-response)